# Notebook 6: Capstone — Passive Hinge Impedance Control

## From Tendon API to Impedance Control Through a Cable

In Notebook 5 we learned how MuJoCo spatial tendons work: site-based routing,
`data.ten_length`, and tension-only behaviour via `springlength`. Now we use these
concepts to build an impedance controller for a more complex system.

### The Capstone Geometry

The capstone model (`tendon_capstone.xml`) connects three bodies through a cable:

```
Block (ceiling rail, x-axis)  ──── cable ──→  Corner pulley (fixed site)
                                                       │
                                                       ▼
                                              Lever tip (wall lever)
```

- A **block** slides along a ceiling rail (prismatic joint, `block_slide`)
- A **fixed corner pulley** at the top-right corner redirects the cable 90 degrees
- A **lever** mounted on the right wall swings about a hinge joint (`lever_hinge`)

**The key pedagogical point:** The controller reads `data.qpos[lever_hinge]` (lever
angle) as the controlled variable but actuates the block. Force is transmitted
**indirectly** through the cable spring: block position → cable stretch → cable tension
→ torque at lever tip. The hinge joint itself has no actuator in this (passive) scenario.

### Passive vs Assisted

This notebook is the **passive** scenario: the hinge has no external torque (`ctrl[1] = 0`).
The controller achieves lever tracking entirely through cable-transmitted force.
In Notebook 7, we add a proportional hinge assist torque and observe improved tracking.

## Setup

In [ ]:
import os
import tempfile
import mujoco
import numpy as np
import matplotlib.pyplot as plt

try:
    import mediapy as media
    HAS_MEDIAPY = True
except ImportError:
    HAS_MEDIAPY = False
    print("mediapy not available — inline rendering disabled")

%matplotlib inline

print(f"MuJoCo version: {mujoco.__version__}")
print(f"NumPy version:  {np.__version__}")

## Load Model and Extract Named Indices

In [ ]:
# Load the shared capstone model (validated in Plan 01)
model = mujoco.MjModel.from_xml_path('../models/tendon_capstone.xml')
data  = mujoco.MjData(model)

print(f"nq={model.nq}, nv={model.nv}, nu={model.nu}, ntendon={model.ntendon}")
print(f"timestep={model.opt.timestep} s")

# Extract named joint indices (robust to model reordering)
lever_qpos_idx = model.joint("lever_hinge").qposadr[0]   # index into data.qpos
lever_vel_idx  = model.joint("lever_hinge").dofadr[0]    # index into data.qvel
block_slide_idx = model.joint("block_slide").qposadr[0]  # index into data.qpos
block_vel_idx  = model.joint("block_slide").dofadr[0]    # index into data.qvel

# Extract named actuator indices
block_ctrl_idx = model.actuator("block_motor").id        # ctrl[0]: block force (N)
hinge_ctrl_idx = model.actuator("hinge_motor").id        # ctrl[1]: hinge torque (Nm)

print(f"\nlever_qpos_idx={lever_qpos_idx}, lever_vel_idx={lever_vel_idx}")
print(f"block_slide_idx={block_slide_idx}, block_vel_idx={block_vel_idx}")
print(f"block_ctrl_idx={block_ctrl_idx}, hinge_ctrl_idx={hinge_ctrl_idx}")

## Read Tendon Parameters from Model

In [ ]:
# Read cable stiffness and natural length from model at runtime
tendon_id       = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_TENDON, "cable")
L_natural       = model.tendon_lengthspring[tendon_id, 1]   # springlength upper bound
CABLE_STIFFNESS = model.tendon_stiffness[tendon_id]          # N/m

print(f"Cable: L_natural={L_natural:.4f} m, stiffness={CABLE_STIFFNESS:.1f} N/m")
print(f"Tension formula: T = {CABLE_STIFFNESS:.0f} * max(0, ten_length - {L_natural:.4f})")

## Theory: Impedance Control Through a Cable Spring

### Impedance Law at the Lever Joint

The desired behaviour is a virtual spring-damper impedance at the lever:

$$\tau_{imp} = K(\theta_d - \theta) + D(\dot{\theta}_d - \dot{\theta})$$

### Two-Level Control Structure

Unlike direct-drive actuation, the cable is a **passive spring** — it cannot directly
command torque. The cable tension arises from block position, not block force.
This creates a two-level control structure:

**Outer loop (lever impedance → desired cable tension):**

$$\tau_{total} = \tau_{imp} + \tau_{gravity}$$
$$T_{desired} = \max\!\left(0,\; \frac{\tau_{total}}{r_{eff}}\right)$$

where $\tau_{gravity}$ = `data.qfrc_bias[lever_dof]` (MuJoCo's gravity torque at the lever),
and $r_{eff} \approx 0.384$ m is the effective moment arm at $\theta=0$.
The $\max(0, \cdot)$ reflects that the cable is tension-only: it cannot push.

**Inner loop (desired cable tension → desired block position → block PD):**

$$L_{desired} = \frac{T_{desired}}{k} + L_{natural}$$

$$x_{block,desired} = 1.0 - (L_{desired} - L_{seg2}(\theta))$$

$$F_{block} = K_{block}(x_{desired} - x_{block}) + D_{block}(0 - \dot{x}_{block})$$

where $L_{seg2}(\theta)$ is the current corner-pulley-to-lever-tip distance,
and `1.0` is the block-anchor-to-corner distance when block is at $x=0$.

### Why Gravity Compensation Is Required

The lever's natural (unforced) equilibrium is at $\approx -53°$ (lever pointing downward).
Without gravity compensation, the cable would need to overcome the full gravitational
restoring force on every step, making positive-theta tracking impossible with
a tension-only cable.

### Block Motor Sign Convention

The `block_slide` joint axis = $+X$ = **toward the corner** (slack direction).
Positive `ctrl[0]` pushes block in $+X$ → cable shortens → **slack**.
The inner loop uses position control, so the sign is handled automatically.

## Controller Implementation

In [ ]:
# Controller parameters
K        = 50.0    # Nm/rad -- lever impedance stiffness
D        = 10.0    # Nm*s/rad -- lever impedance damping
r_lever  = 0.6     # m -- lever arm length (hinge to lever_tip)
R_EFF    = 0.384   # m -- effective moment arm at theta=0 (cable force perpendicular component)

# Inner-loop block position PD gains
K_BLOCK  = 2000.0  # N/m -- block position stiffness
D_BLOCK  = 50.0    # N*s/m -- block velocity damping


def capstone_controller(model, data, q_des, dq_des, tau_hinge=0.0):
    """Two-level impedance controller for the capstone tendon-lever system.

    Outer loop: lever angle impedance -> desired cable tension.
    Inner loop: cable tension -> desired cable length -> desired block position -> block PD.

    Args:
        model:      MjModel instance
        data:       MjData instance (read-write: sets data.ctrl)
        q_des:      desired lever angle (rad)
        dq_des:     desired lever angular velocity (rad/s)
        tau_hinge:  hinge motor torque (Nm): 0.0 = passive, >0 = assisted
    """
    # --- OUTER LOOP: lever impedance ---
    q  = data.qpos[lever_qpos_idx].copy()   # current lever angle
    dq = data.qvel[lever_vel_idx].copy()    # current lever angular velocity

    # Impedance torque + gravity compensation at the lever joint
    # qfrc_bias includes gravitational forces that would accelerate the lever downward
    tau_imp   = K * (q_des - q) + D * (dq_des - dq)
    tau_grav  = data.qfrc_bias[lever_vel_idx]     # gravity torque to counteract
    tau_total = tau_imp + tau_grav

    # Required cable tension (cable is tension-only: clip at 0)
    T_desired = max(0.0, tau_total / R_EFF)

    # --- INNER LOOP: block position PD ---
    # Required cable length to achieve T_desired via passive cable spring
    L_desired = T_desired / CABLE_STIFFNESS + L_natural

    # Corner-pulley-to-lever-tip distance at current lever angle (segment 2)
    tip_x = 1.0 - r_lever * np.cos(q)
    tip_z = 0.5 + r_lever * np.sin(q)
    seg2  = np.sqrt((tip_x - 1.0)**2 + (tip_z - 1.0)**2)

    # Desired block position (from cable geometry: seg1 = 1.0 - block_x)
    seg1_desired  = L_desired - seg2
    block_x_des   = np.clip(1.0 - seg1_desired, -1.0, 0.5)

    # PD control on block position
    block_x  = data.qpos[block_slide_idx].copy()
    block_xd = data.qvel[block_vel_idx].copy()
    F_block  = K_BLOCK * (block_x_des - block_x) + D_BLOCK * (0.0 - block_xd)

    # Apply actuator commands
    data.ctrl[block_ctrl_idx] = np.clip(F_block, -100.0, 100.0)
    data.ctrl[hinge_ctrl_idx] = tau_hinge


def get_cable_tension(data, tendon_idx=0):
    """Compute analytical cable tension from current tendon length.

    Note: data.ten_force does not exist in MuJoCo 3.6.0.
    Tension is computed analytically from data.ten_length.

    Returns: cable tension in Newtons (zero when cable is slack).
    """
    stretch = float(data.ten_length[tendon_idx]) - L_natural
    return CABLE_STIFFNESS * max(0.0, stretch)


print("Controller ready.")
print(f"  Lever impedance: K={K} Nm/rad, D={D} Nm*s/rad")
print(f"  Block PD:        K_BLOCK={K_BLOCK} N/m, D_BLOCK={D_BLOCK} N*s/m")
print(f"  Effective moment arm R_EFF={R_EFF} m")

## Experiment: Sinusoidal Trajectory Tracking (Passive Hinge)

### Desired Trajectory

$$\theta_d(t) = A \sin(2\pi f t), \quad \dot{\theta}_d(t) = A \cdot 2\pi f \cos(2\pi f t)$$

with $A = 0.3$ rad ($\approx 17°$) and $f = 0.5$ Hz.

### Passive Condition

The hinge motor torque is zero: $\tau_{hinge} = 0$ (passive scenario). All lever
torque is transmitted through the cable.

In [ ]:
# Trajectory parameters
A_des   = 0.3          # rad (~17 degrees)
f_des   = 0.5          # Hz
omega_d = 2 * np.pi * f_des

# Simulation parameters
sim_duration = 5.0
dt           = model.opt.timestep
n_steps      = int(sim_duration / dt)

TAU_HINGE_PASSIVE = 0.0  # no hinge torque for passive scenario

# Reset and initialise
mujoco.mj_resetData(model, data)
mujoco.mj_forward(model, data)

# Pre-allocate recording arrays (record-then-plot convention)
t_hist  = np.zeros(n_steps)
q_hist  = np.zeros(n_steps)   # actual lever angle (rad)
qd_hist = np.zeros(n_steps)   # desired lever angle (rad)
F_hist  = np.zeros(n_steps)   # cable tension (N)

# Simulation loop
for i in range(n_steps):
    t = data.time

    # Desired trajectory
    q_des  = A_des * np.sin(omega_d * t)
    dq_des = A_des * omega_d * np.cos(omega_d * t)

    # Record BEFORE stepping
    t_hist[i]  = t
    q_hist[i]  = data.qpos[lever_qpos_idx].copy()
    qd_hist[i] = q_des
    F_hist[i]  = get_cable_tension(data)

    # Apply controller
    capstone_controller(model, data, q_des, dq_des, tau_hinge=TAU_HINGE_PASSIVE)

    # Advance simulation
    mujoco.mj_step(model, data)

# Compute RMS tracking error
rms_rad = np.sqrt(np.mean((q_hist - qd_hist)**2))
rms_deg = np.degrees(rms_rad)
PASSIVE_RMS_DEG = rms_deg   # save for NB7 reference

print(f"Passive scenario simulation: {sim_duration} s ({n_steps} steps)")
print(f"  Lever range: [{np.degrees(q_hist.min()):.1f}, {np.degrees(q_hist.max()):.1f}] degrees")
print(f"  Max cable tension: {F_hist.max():.2f} N")
print()
print(f"Passive scenario RMS tracking error: {rms_deg:.2f} degrees")

assert rms_deg < 5.0, (
    f"RMS tracking error {rms_deg:.2f} deg exceeds 5 deg threshold."
)
print(f"  PASS: RMS < 5 degrees")

## Results: Angle Tracking and Cable Tension

The two-panel figure shows:
1. **Lever angle tracking** — desired (dashed) vs actual (solid)
2. **Cable tension** — the cable tension oscillates as the controller adjusts block position

The cable is tension-only, so tension is zero whenever the desired torque is negative
(i.e., when the controller wants to push the lever downward).

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)

# --- Top panel: Lever angle tracking ---
ax = axes[0]
ax.plot(t_hist, np.degrees(qd_hist), 'k--', lw=1.5, label='Desired $\\theta_d$')
ax.plot(t_hist, np.degrees(q_hist),  'b-',  lw=1.5, label='Actual $\\theta$')
ax.set_ylabel('Lever angle (degrees)')
ax.set_title('Passive Hinge — Lever Angle Tracking')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.annotate(
    f'RMS = {rms_deg:.2f}°',
    xy=(t_hist[-1] * 0.72, np.degrees(A_des) * 0.80),
    fontsize=11, color='darkblue',
    bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', edgecolor='gray', alpha=0.9)
)

# --- Bottom panel: Cable tension ---
ax = axes[1]
ax.plot(t_hist, F_hist, 'r-', lw=1.5, label='Cable tension T (N)')
ax.fill_between(t_hist, F_hist, 0, alpha=0.15, color='red')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Cable tension (N)')
ax.set_title('Cable Tension')
ax.set_ylim(bottom=0)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Notebook 6: Capstone Passive Hinge Impedance Control', fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = os.path.join(tempfile.gettempdir(), 'nb6_passive_results.png')
plt.savefig(save_path, dpi=100, bbox_inches='tight')
plt.show()
print(f"Figure saved to {save_path}")

## Inline Video: Passive Impedance Tracking

The video below re-runs the sinusoidal tracking simulation and renders the capstone mechanism in 3D. You should see the block sliding along the ceiling rail, the cable transmitting force through the corner pulley, and the lever swinging to follow the desired trajectory.

Tendon rendering is enabled so the cable path is visible throughout the motion.

In [ ]:
# --- Inline Video: Passive hinge impedance tracking ---
try:
    renderer = mujoco.Renderer(model, height=360, width=480)

    # Enable tendon rendering in scene options
    scene_opt = mujoco.MjvOption()
    scene_opt.flags[mujoco.mjtVisFlag.mjVIS_TENDON] = True

    # Trajectory parameters (same as experiment above)
    A_vid   = 0.3
    f_vid   = 0.5
    omega_vid = 2 * np.pi * f_vid
    dt_vid  = model.opt.timestep

    # Reset simulation
    mujoco.mj_resetData(model, data)
    mujoco.mj_forward(model, data)

    frames = []
    frame_every = 10
    duration_vid = 5.0
    n_vid = int(duration_vid / dt_vid)

    for i in range(n_vid):
        t = data.time
        q_des  = A_vid * np.sin(omega_vid * t)
        dq_des = A_vid * omega_vid * np.cos(omega_vid * t)
        capstone_controller(model, data, q_des, dq_des, tau_hinge=0.0)
        mujoco.mj_step(model, data)
        if i % frame_every == 0:
            renderer.update_scene(data, scene_option=scene_opt)
            frames.append(renderer.render())

    renderer.close()
    fps = int(1.0 / (dt_vid * frame_every))

    if HAS_MEDIAPY:
        media.show_video(frames, fps=fps)
    print(f'Video: {len(frames)} frames at {fps} fps')
except Exception as e:
    print(f'Renderer not available in this environment: {e}')
    print('This is expected when running headless (e.g., nbconvert).')

## Optional: Interactive Passive Viewer

In [ ]:
# Uncomment to open interactive viewer (non-blocking on Windows).
# Requires `pip install mujoco` with viewer support.

# mujoco.mj_resetData(model, data)
# mujoco.viewer.launch_passive(model, data)

## Summary

### Key Results

| Metric | Value |
|--------|-------|
| Trajectory amplitude | 0.3 rad (17°), 0.5 Hz |
| Controller | Two-level: lever impedance (K=50, D=10) + block position PD |
| Passive RMS error | see output above |
| Threshold | < 5° |

### What We Built

1. **Two-level controller:** Outer loop computes desired cable tension from lever
   impedance; inner loop converts to desired cable length and then desired block
   position using inverse kinematics of the cable geometry.
2. **Gravity compensation:** `data.qfrc_bias[lever_dof]` provides the gravitational
   bias torque that must be added to the impedance torque so the cable need only
   counteract tracking errors, not fight gravity continuously.
3. **Tension-only cable:** The cable clamps at $T_{desired} = 0$ when the impedance
   law calls for a downward (compressive) force; gravity provides the restoring torque.

---

## What's Next: Notebook 7 — Assisted Hinge

In this passive scenario the cable alone handles lever tracking while gravity acts
as a passive restoring force. **What happens when the hinge motor provides an
assisting torque proportional to the impedance error?** In Notebook 7 we add
$\tau_{hinge} = \alpha \cdot \tau_{imp}$ and observe how the shared load between
cable and hinge motor reduces tracking error.